In [1]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay
import joblib

In [2]:
# Load 4 selected CSVs
benign = pd.read_csv('../data/Monday-WorkingHours.pcap_ISCX.csv')
attack_ddos = pd.read_csv('../data/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv')
attack_portscan = pd.read_csv('../data/Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv')
attack_web = pd.read_csv('../data/Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv')

# Combine into one dataframe
data = pd.concat([benign, attack_ddos, attack_portscan, attack_web], ignore_index=True)

# Clean column names
data.columns = data.columns.str.strip()

# Encode labels: BENIGN=0, ATTACK=1
data['Label'] = data['Label'].apply(lambda x: 0 if x == 'BENIGN' else 1)

# Check label distribution
print(data['Label'].value_counts())

Label
0    923359
1    289137
Name: count, dtype: int64


In [3]:
# Separate features and target
X = data.drop('Label', axis=1)
y = data['Label']

# Replace infinities and NaNs
X.replace([np.inf, -np.inf], np.nan, inplace=True)
X.fillna(0, inplace=True)

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [4]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
svm_model = SVC(kernel='linear', probability=True, random_state=42)
svm_model.fit(X_train, y_train)

In [ ]:
# Predict and evaluate
y_pred = svm_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print("SVM Accuracy:", accuracy)

In [ ]:
# Display confusion matrix
ConfusionMatrixDisplay.from_estimator(svm_model, X_test, y_test)
plt.show()

In [ ]:
if svm_model.kernel == 'linear':
    coef = pd.Series(svm_model.coef_[0], index=X.columns).sort_values(key=abs, ascending=False).head(10)
    plt.figure(figsize=(10,6))
    coef.plot(kind='barh', color='skyblue')
    plt.xlabel('Coefficient Magnitude')
    plt.ylabel('Feature')
    plt.title('Top 10 Features (Linear SVM)')
    plt.gca().invert_yaxis()
    plt.show()
else:
    print("Feature importance not available for non-linear kernels")

In [ ]:
# Save SVM model
import os
os.makedirs('./models', exist_ok=True)
joblib.dump(svm_model, './models/svm_model.pkl')
print("✓ SVM model saved to ./models/svm_model.pkl")